# Auto-encodeurs débruiteurs

## Vérification de l'utilisation de GPU

Allez dans le menu `Exécution > Modifier le type d'execution` et vérifiez que l'on est bien en Python 3 et que l'accélérateur matériel est configuré sur « GPU ».

In [ ]:
!nvidia-smi

## Installation et import de PyTorch Lightning et des autres librairies nécessaires

In [ ]:
!pip install -q lightning

In [ ]:
import lightning
import matplotlib.pyplot as plt
import torch
import torchvision
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

## Chargement de MNIST

Nous allons utiliser un prétraîtement légèrement différent des autres fois : étant donné que nous voulons pouvoir prédire les valeurs données en entrée en sortie (principe de l'auto-encodage), nous allons simplement projeter ces valeurs dans $[0, 1]$ au lieu de $[0, 255]$. Notez qu'habituellement nous ne faisons pas ça : nous normalisons en centrant sur zéro et en divisant par l'écart-type.

In [ ]:
train_data = torchvision.datasets.MNIST("data", train=True, download=True)
test_data = torchvision.datasets.MNIST("data", train=False, download=True)
nb_classes = 10
input_dim = 28 * 28
X_train = train_data.data.reshape(-1, input_dim).float() / 255.0
X_test = test_data.data.reshape(-1, input_dim).float() / 255.0
y_test = test_data.targets

## Application d'un bruit  gaussien

In [ ]:
noise_factor = 0.5
X_train_noisy = X_train + torch.normal(0., noise_factor, X_train.shape)
X_test_noisy = X_test + torch.normal(0., noise_factor, X_test.shape)

# On clip les valeurs pour éviter les pixels plus blanc que blanc (ou plus noir
# que noir)
X_train_noisy.clamp_(0, 1)
X_test_noisy.clamp_(0, 1)

In [ ]:
n = 10
f, ax = plt.subplots(1, n, figsize=(n * 1.4, 2))
for i in range(n):
  ax[i].imshow(X_test_noisy[i].reshape(28, 28), cmap="gray_r")
  ax[i].set_title(int(y_test[i]))
  ax[i].axis("off")
plt.show()

## Création de l'autoencodeur débruiteur



In [ ]:
class AutoEncoder(lightning.LightningModule):
  """Lightning wrapper: reconstruct the clean image from the noisy one."""

  def __init__(self,
               model: nn.Module,
               loss: nn.Module,
               learning_rate: float = 1e-3,
               input_shape: tuple[int, ...] = (input_dim,)) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model", "loss"])
    self.model = model
    self.loss = loss
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, *input_shape)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor],
            stage: str) -> torch.Tensor:
    noisy_images, images = batch
    loss = self.loss(self(noisy_images), images)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(),
                            lr=self.hparams.learning_rate)


@torch.no_grad()
def predict(model: nn.Module,
            X: torch.Tensor,
            batch_size: int = 256) -> torch.Tensor:
  """Apply a network to every row of X, batch by batch."""
  device = next(model.parameters()).device
  model.eval()
  return torch.cat([model(batch.to(device)).cpu()
                    for batch in X.split(batch_size)])

In [ ]:
# Votre code ici
encoding_dim = 12

autoencoder = nn.Sequential()

### Solution

In [ ]:
encoding_dim = 12

autoencoder = nn.Sequential(
    nn.Linear(input_dim, encoding_dim),
    nn.ReLU(),
    nn.Linear(encoding_dim, input_dim),
    nn.Sigmoid())

model = AutoEncoder(autoencoder, loss=nn.MSELoss())

print(ModelSummary(model, max_depth=-1))

## Apprentissage

*Écrivez les lignes correspondant à l'apprentissage de votre autoencodeur :*

- *50 itérations devraient suffire*
- *Utilisez un batch de 256*

In [ ]:
# Votre code ici

##Solution

In [ ]:
batch_size = 256

# 20 % des données d'entraînement sont mises de côté pour la validation
train_dataset, val_dataset = random_split(
    TensorDataset(X_train_noisy, X_train), [0.8, 0.2])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

trainer = lightning.Trainer(max_epochs=50,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="autoencoder"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

## Base de Test

Autoencodez les images de test et stockez les images obtenues dans la variable `X_test_noisy_pred`

In [ ]:
# Votre code ici
X_test_noisy_pred = X_test_noisy

### Solution

In [ ]:
X_test_noisy_pred = predict(autoencoder, X_test_noisy)

## Affichage visuel de la performance

In [ ]:
n = 10
_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for i, (ax_top, ax_bottom) in enumerate(ax.T):
  # L'original en haut
  ax_top.imshow(X_test_noisy[i].reshape(28, 28), cmap="gray_r")
  ax_top.set_title(str(int(y_test[i])))
  ax_top.axis("off")

  # La reconstruction en bas
  ax_bottom.imshow(X_test_noisy_pred[i].reshape(28, 28), cmap="gray_r")
  ax_bottom.axis("off")
plt.show()

## Essayez avec plus de neurones

Que se passe-t-il ?

## Utilisation des réseaux convolutifs

Pour cela il faut remettre chaque image sous forme 1 × 28 × 28. Les CNNs ont besoin de cette dimension de canaux, que PyTorch attend juste après celle de batch (il pourrait y avoir plus de canaux que le niveau de gris : il y en a 3 pour les images en couleur et plus encore dans les couches intermédiaires d'un réseau convolutif où le nombre de canaux en entrée d'une couche sera le nombre de kernels de la couche précédente).

In [ ]:
X_train = X_train.reshape(-1, 1, 28, 28)
X_test = X_test.reshape(-1, 1, 28, 28)
X_train_noisy = X_train_noisy.reshape(-1, 1, 28, 28)
X_test_noisy = X_test_noisy.reshape(-1, 1, 28, 28)

## Création du modèle

On utilisera des séquences de [`Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html), [`MaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) avec des kernel de respectivement 3 × 3 et 2 × 2 pour la partie encodeur. Dans la convolution, il faudra utiliser l'option `padding="same"` afin d'éviter les effets de bord (l'image étant déjà assez petite comme ça).

Pour la partie décodeur, on utilisera des [`Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) de même nature suivis par des [`Upsample`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html) (de facteur 2) qui correspondent à l'opération inverse de [`MaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html).

N'hésitez pas à abuser de `ModelSummary` pour vous y retrouver. L'objectif étant de retrouver une image 1 × 28 × 28 à la sortie du dernier [`Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html). En effet, finir par un [`Upsample`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html) serait une mauvaise idée.

Le réseau va être profond, on utilisera des fonctions d'activation ReLU, sauf pour la dernière où on utilisera une sigmoïde.

In [ ]:
# Votre code ici
autoencoder = nn.Sequential()

### Solution

In [ ]:
encoding_dim = 20


def conv(in_channels: int,
         out_channels: int,
         activation: type[nn.Module] = nn.ReLU) -> nn.Sequential:
  return nn.Sequential(
      nn.Conv2d(in_channels, out_channels, (3, 3), padding="same"),
      activation())


def max_pool() -> nn.MaxPool2d:
  return nn.MaxPool2d((2, 2), ceil_mode=True)


def up_sampling() -> nn.Upsample:
  return nn.Upsample(scale_factor=2)


autoencoder = nn.Sequential(
    # Encodage
    conv(1, 32),
    max_pool(),
    conv(32, 32),
    max_pool(),
    conv(32, 1),
    nn.Flatten(),
    nn.Linear(7 * 7, encoding_dim),

    # Décodage
    nn.Linear(encoding_dim, 7 * 7),
    nn.Unflatten(1, (1, 7, 7)),
    conv(1, 32),
    up_sampling(),
    conv(32, 32),
    up_sampling(),
    conv(32, 1, nn.Sigmoid))

model = AutoEncoder(autoencoder,
                    loss=nn.BCELoss(),
                    learning_rate=1e-4,
                    input_shape=X_train.shape[1:])

print(ModelSummary(model, max_depth=-1))

## Apprentissage

*Écrivez les lignes correspondant à l'apprentissage de votre autoencodeur :*

- *100 itérations devraient suffire*
- *Utilisez un batch de 256*

In [ ]:
# Votre code ici

### Solution

In [ ]:
batch_size = 128

train_dataset, val_dataset = random_split(
    TensorDataset(X_train_noisy, X_train), [0.8, 0.2])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

trainer = lightning.Trainer(max_epochs=100,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="conv_autoencoder"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

## Affichage des performances

In [ ]:
X_test_noisy_pred = predict(autoencoder, X_test_noisy).reshape(-1, 28, 28)

n = 10

random_indexes = torch.randperm(X_test.shape[0])[:n]

_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for (ax_top, ax_bottom), random_index in zip(ax.T, random_indexes):
  # L'image originale en haut
  ax_top.set_title(str(int(y_test[random_index])))
  ax_top.imshow(X_test_noisy[random_index].reshape(28, 28), cmap="gray_r")
  ax_top.axis("off")

  # L'image reconstruite en bas
  ax_bottom.imshow(X_test_noisy_pred[random_index], cmap="gray_r")
  ax_bottom.axis("off")
plt.show()

## Sur des images non-bruitées

In [ ]:
X_test_pred = predict(autoencoder, X_test).reshape(-1, 28, 28)

n = 10

random_indexes = torch.randperm(X_test.shape[0])[:n]

_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for (ax_top, ax_bottom), random_index in zip(ax.T, random_indexes):
  # L'image originale en haut
  ax_top.set_title(str(int(y_test[random_index])))
  ax_top.imshow(X_test[random_index].reshape(28, 28), cmap="gray_r")
  ax_top.axis("off")

  # L'image reconstruite en bas
  ax_bottom.imshow(X_test_pred[random_index], cmap="gray_r")
  ax_bottom.axis("off")
plt.show()